# EdgeGuard-Road · oturum kaybına dayanıklı bilimsel kampanya

Tek kontrol hücresinde hedefi seçip **Run all** kullanın. Yeni Colab oturumu `/content` alanını sıfırdan kurar, Drive'daki doğrulanmış küçük kampanya durumunu ve son checkpoint generation'ını geri yükler, yalnız eksik aşamaları çalıştırır. Gerçek veri olmadan bilimsel metrik üretmez.

In [ ]:
import json
import os
import sys
from pathlib import Path

LOCAL_TEST_MODE = os.environ.get("EDGEGUARD_NOTEBOOK_LOCAL_TEST") == "1"
if LOCAL_TEST_MODE:
    PROJECT_ROOT = Path(os.environ.get("EDGEGUARD_PROJECT_ROOT", Path.cwd())).resolve()
    DRIVE_ROOT = Path(os.environ["EDGEGUARD_TEST_DRIVE_ROOT"]).resolve()
    CONTENT_ROOT = Path(os.environ["EDGEGUARD_TEST_CONTENT_ROOT"]).resolve()
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    CONTENT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/edgeguard-road")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    CONTENT_ROOT = Path("/content")

REPOSITORY = "https://github.com/emrealmaoglu/edgeguard-road.git"
BRANCH = "stabilize/colab-v2"
EXPECTED_PROJECT_COMMIT = "5f665cdbe0caad011ff66ad7b210ab8121d9fad1"
LOCAL_DATA_ROOT = CONTENT_ROOT / "edgeguard-data"
CITYSCAPES_ROOT = LOCAL_DATA_ROOT / "cityscapes"
BDD100K_ROOT = LOCAL_DATA_ROOT / "bdd100k"
IDD20K_ROOT = LOCAL_DATA_ROOT / "idd20k"
ACDC_ROOT = LOCAL_DATA_ROOT / "acdc"
DATASET_ROOTS = {
    "cityscapes": CITYSCAPES_ROOT,
    "bdd100k": BDD100K_ROOT,
    "idd20k": IDD20K_ROOT,
    "acdc": ACDC_ROOT,
}
SCIENTIFIC_SOURCE_DATASETS = ["cityscapes", "idd20k"]
SECONDARY_SCIENTIFIC_DATASETS = [
    dataset for dataset in SCIENTIFIC_SOURCE_DATASETS if dataset != "cityscapes"
]
OFFICIAL_VALIDATION_DATASETS = list(SCIENTIFIC_SOURCE_DATASETS)
PROVISIONAL_ENGINEERING_DATASETS = ["bdd100k"]
STAGE_PROVISIONAL_BDD = False  # Yalnız açık mühendislik audit'i için True.
OPTIONAL_EVALUATION_DATASETS = []  # Model freeze sonrası ör. ["acdc"]
WORK_ROOT = CONTENT_ROOT / "edgeguard-work"
CAMPAIGN_ID = "semantic-cs-idd-v2"
CAMPAIGN_TARGET = "audit"  # audit|smoke|pilot|screening|hpo|final|evaluate|export|report
RUN_STAGE = CAMPAIGN_TARGET  # Legacy report naming; execution uses TRAINING_STAGES.
AUTO_RESUME = True
DEEP_VERIFY_ARCHIVES = False
ALLOW_FINAL_DATA = False
CORE_MODELS = ["segformer_b0", "fast_scnn", "pidnet_s"]
EXTENSION_MODELS = ["ddrnet_23_slim", "bisenetv2"]
RUN_MODELS = (
    CORE_MODELS if CAMPAIGN_TARGET in {"smoke", "pilot"} else CORE_MODELS + EXTENSION_MODELS
)
ALLOW_INELIGIBLE_BDD_SMOKE = True  # Provisional audit için; DATA_MANIFESTS listesine girmez.
RUN_DATA_STAGING = not LOCAL_TEST_MODE
RUN_MULTIDOMAIN_AUDIT = not LOCAL_TEST_MODE
RUN_FREEZE = False  # Cityscapes + IDD candidate manifestleri incelendikten sonra True.
RUN_SOURCE_VALIDATION_AUDIT = False
RUN_FREEZE_SOURCE_VALIDATION = False
MANIFEST_REVIEW_RECEIPT_ROOT = WORK_ROOT / "reviews/manifest-freeze"
RELEASE_CANDIDATE = WORK_ROOT / "accepted_release.candidate.json"
RELEASE_REVIEW_RECEIPT = WORK_ROOT / "reviews/release.review.json"
ACCEPTED_RELEASE = WORK_ROOT / "accepted_release.json"
RUN_ACCEPT_RELEASE = False  # İnsan review receipt'i hazırlandıktan sonra açıkça True.
FINAL_MODELS = []
RUN_ACDC = False
RUN_SHIFT_CALIBRATION = False
RUN_PERCEPTION_PREVIEW = False
PREVIEW_IMAGE = Path("/content/preview.png")
CREATE_REVIEW_PACKAGE = True
DOWNLOAD_REVIEW_PACKAGE = False  # True: küçük rapor/şekil ZIP'i tarayıcıya indirir.
DOWNLOAD_LATEST_FAILURE_REPORT = False  # Hata sonrası son hücreyi bununla yeniden çalıştırın.
RUN_SEALED_PACKAGE = False
SEALED_MANIFEST = WORK_ROOT / "manifests/wilddash2.frozen.json"
SEALED_RELEASE = WORK_ROOT / "manifests/wilddash2.release.json"


def persist_bootstrap_failure(notebook, stage, error):
    import json
    import re
    import traceback
    import uuid
    from datetime import datetime, timezone
    from zipfile import ZIP_DEFLATED, ZipFile

    failed_at = datetime.now(timezone.utc)
    failure_id = f"{failed_at.strftime('%Y%m%dT%H%M%S.%fZ')}-{stage}-{uuid.uuid4().hex[:8]}"
    root = DRIVE_ROOT / "EdgeGuard/failures/bootstrap" / failure_id
    root.mkdir(parents=True, exist_ok=False)
    rendered = "".join(traceback.format_exception(type(error), error, error.__traceback__))
    rendered = re.sub(r"(?i)(token|password|secret|api[_-]?key)=\S+", r"\1=<redacted>", rendered)
    payload = {
        "record_type": "edgeguard_colab_bootstrap_failure",
        "failure_id": failure_id,
        "failed_at": failed_at.isoformat(),
        "notebook": notebook,
        "stage": stage,
        "error_type": type(error).__name__,
        "traceback": rendered,
    }
    report = root / "failure.json"
    report.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    package = root / "failure-report.zip"
    with ZipFile(package, "w", compression=ZIP_DEFLATED) as archive:
        archive.write(report, arcname="failure.json")
    print("EDGEGUARD BOOTSTRAP FAILURE:", package)
    return package

In [ ]:
FINAL_PROTOCOL_CODE = r"""# Accepted-release evaluation/export/report is owned by
# scripts/colab_pipeline.py.
print("Accepted-release sonrası fazlar sürümlenmiş Python orchestrator tarafından işlendi.")
"""

In [ ]:
import subprocess


def run_bootstrap_command(command):
    completed = subprocess.run(command, capture_output=True, text=True)
    output = "\n".join(
        value.strip() for value in (completed.stdout, completed.stderr) if value.strip()
    )
    if output:
        print(output)
    if completed.returncode != 0:
        raise RuntimeError(
            f"Bootstrap command failed with exit code {completed.returncode}: {command}\n"
            + output[-8000:]
        )
    return completed


try:
    if LOCAL_TEST_MODE:
        print("LOCAL_TEST_MODE: mount, clone ve kurulum atlandı.")
    elif not (PROJECT_ROOT / ".git").is_dir():
        run_bootstrap_command(["git", "clone", "--branch", BRANCH, REPOSITORY, str(PROJECT_ROOT)])
    else:
        run_bootstrap_command(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", BRANCH])
        run_bootstrap_command(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH])
        run_bootstrap_command(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"])
except BaseException as error:
    persist_bootstrap_failure("EdgeGuard_Road_Colab.ipynb", "git-clone-or-update", error)
    raise
if not LOCAL_TEST_MODE and EXPECTED_PROJECT_COMMIT:
    run_bootstrap_command(["git", "-C", str(PROJECT_ROOT), "checkout", EXPECTED_PROJECT_COMMIT])
PROJECT_COMMIT = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if EXPECTED_PROJECT_COMMIT and PROJECT_COMMIT != EXPECTED_PROJECT_COMMIT:
    if not LOCAL_TEST_MODE:
        raise RuntimeError("Project commit does not match EXPECTED_PROJECT_COMMIT")
    print(
        "LOCAL_TEST_MODE: sabit Colab commit checkout edilmedi; "
        f"yerel HEAD={PROJECT_COMMIT[:12]}, beklenen={EXPECTED_PROJECT_COMMIT[:12]}."
    )
# Never retain imports from a prior checkout in a reused Colab kernel.
for loaded_module in list(sys.modules):
    if loaded_module == "edgeguard" or loaded_module.startswith("edgeguard."):
        del sys.modules[loaded_module]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from edgeguard.rescue.colab_failures import (  # noqa: E402
    ColabFailureReporter,
    run_logged_command,
)
from edgeguard.rescue.colab_recovery import (  # noqa: E402
    action_requirements,
    completion_is_valid,
    latest_checkpoint,
    quarantine_incomplete,
    write_completion_receipt,
)
from edgeguard.serialization import canonical_json, sha256_file, sha256_payload  # noqa: E402

ACTION_PLAN = action_requirements(
    CAMPAIGN_TARGET,
    allow_final_data=ALLOW_FINAL_DATA,
    provisional_bdd=STAGE_PROVISIONAL_BDD,
)
if RUN_ACDC and ALLOW_FINAL_DATA and "acdc" not in ACTION_PLAN["datasets"]:
    ACTION_PLAN["datasets"].append("acdc")
PLANNED_STAGES = ACTION_PLAN["stages"]
TRAINING_STAGES = ACTION_PLAN["training_stages"]
RUNTIME_REQUIRED = ACTION_PLAN["runtime_required"]
RUN_HPO = "hpo" in PLANNED_STAGES
RUN_FINAL_EVALUATION = "evaluate" in PLANNED_STAGES
if CAMPAIGN_TARGET in {"evaluate", "export", "report"} and not ALLOW_FINAL_DATA:
    raise RuntimeError(
        "Source/external final data is sealed. First complete CAMPAIGN_TARGET='final'; "
        "then provide ACCEPTED_RELEASE and set ALLOW_FINAL_DATA=True."
    )
RUN_MULTIDOMAIN_AUDIT = RUN_MULTIDOMAIN_AUDIT and "audit" in PLANNED_STAGES
RUN_DATA_STAGING = RUN_DATA_STAGING and bool(ACTION_PLAN["datasets"])
if (TRAINING_STAGES or RUN_HPO or RUN_FINAL_EVALUATION) and RUN_FREEZE:
    print("RUN_FREEZE açık: yalnız hash-bağlı insan review receipt'leri kullanılacak.")
print("EDGEGUARD ACTION PLAN:", ACTION_PLAN)

FAILURE_REPORTER = ColabFailureReporter(
    DRIVE_ROOT / "EdgeGuard/failures" / CAMPAIGN_ID / PROJECT_COMMIT,
    notebook="EdgeGuard_Road_Colab.ipynb",
    project_commit=PROJECT_COMMIT,
    context={"branch": BRANCH, "campaign_id": CAMPAIGN_ID, "local_test_mode": LOCAL_TEST_MODE},
)
FAILURE_REPORTER.add_diagnostic_root("runtime-evidence", CONTENT_ROOT / "edgeguard-evidence")
FAILURE_REPORTER.add_diagnostic_root("runtime-logs", CONTENT_ROOT / "edgeguard-logs")
COMMAND_LOG_ROOT = CONTENT_ROOT / "edgeguard-command-logs"
FAILURE_REPORTER.add_diagnostic_root("command-logs", COMMAND_LOG_ROOT)
FAILURE_REPORTER.add_diagnostic_root("work", WORK_ROOT)
FAILURE_REPORTER.install_ipython_hook()


def run_colab_command(command, *, check=True):
    command_env = os.environ.copy()
    project_src = str(PROJECT_ROOT / "src")
    command_env["PYTHONPATH"] = project_src + os.pathsep + command_env.get("PYTHONPATH", "")
    command_env["UV_CACHE_DIR"] = str(CONTENT_ROOT / "edgeguard-cache/uv")
    return run_logged_command(
        command,
        log_root=COMMAND_LOG_ROOT,
        stage=FAILURE_REPORTER.stage,
        check=check,
        cwd=PROJECT_ROOT,
        env=command_env,
    )


if not LOCAL_TEST_MODE:
    FAILURE_REPORTER.set_stage("project-install")
    run_colab_command([sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[colab]"])
print({"project_commit": PROJECT_COMMIT, "python": sys.version})

In [ ]:
# Önce küçük kampanya durumunu geri yükle, sonra yalnız hedefin gerektirdiği datasetleri stage et.
FAILURE_REPORTER.set_stage("campaign-state-restore-and-conditional-staging")
CAMPAIGN_ROOT = DRIVE_ROOT / "EdgeGuard/campaigns" / CAMPAIGN_ID
RECOVERY_ROOT = CAMPAIGN_ROOT / "recovery/v2"
STATUS_PATH = CAMPAIGN_ROOT / "state/status.json"
run_colab_command(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
        "package-interruption",
        "--status",
        str(STATUS_PATH),
        "--failure-root",
        str(CAMPAIGN_ROOT / "failures"),
    ],
)
LOCAL_STATE_ARCHIVE = CONTENT_ROOT / "edgeguard-campaign-state.tar.gz"
STATE_INCLUDE = [
    "accepted_release.candidate.json",
    "accepted_release.json",
    "audit",
    "calibration",
    "evaluation",
    "external-package",
    "ledger",
    "manifests",
    "multidomain-statistics",
    "preview",
    "reports",
    "reviews",
    "runs",
]
run_colab_command(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
        "cleanup-incoming",
        "--store-root",
        str(RECOVERY_ROOT),
    ],
)
state_pointer = RECOVERY_ROOT / "pointers/campaign-state.json"
if AUTO_RESUME and not WORK_ROOT.exists() and state_pointer.is_file():
    restore_state = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
        "restore",
        "--store-root",
        str(RECOVERY_ROOT),
        "--artifact-id",
        "campaign-state",
        "--destination",
        str(LOCAL_STATE_ARCHIVE),
    ]
    restored = run_colab_command(restore_state, check=False)
    if restored.returncode == 0:
        run_colab_command(
            [
                sys.executable,
                str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
                "restore-state",
                "--archive",
                str(LOCAL_STATE_ARCHIVE),
                "--destination",
                str(WORK_ROOT),
            ],
        )
        print("Drive campaign state restored and verified.")
    else:
        print("Doğrulanmış önceki campaign state yok; temiz kampanya başlatılıyor.")
elif AUTO_RESUME and not WORK_ROOT.exists():
    print("İlk kampanya oturumu: geri yüklenecek state generation yok.")

if RUN_DATA_STAGING and ACTION_PLAN["datasets"]:
    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/prepare_colab_data.py"),
        "--drive-root",
        str(DRIVE_ROOT),
        "stage",
        "--local-root",
        str(LOCAL_DATA_ROOT),
    ]
    datasets_to_stage = [dataset for dataset in ACTION_PLAN["datasets"] if dataset in DATASET_ROOTS]
    for dataset in datasets_to_stage:
        command.extend(["--dataset", dataset])
    if STAGE_PROVISIONAL_BDD:
        command.append("--allow-ineligible")
    run_colab_command(command)


def sync_work_snapshot(label: str) -> None:
    # Only small state is compressed. Checkpoints/ONNX use immutable recovery objects.
    if not WORK_ROOT.exists():
        return
    pack = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
        "pack-state",
        "--work-root",
        str(WORK_ROOT),
        "--output",
        str(LOCAL_STATE_ARCHIVE),
    ]
    for relative in STATE_INCLUDE:
        pack.extend(["--include", relative])
    run_colab_command(pack)
    run_colab_command(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
            "publish",
            "--source",
            str(LOCAL_STATE_ARCHIVE),
            "--store-root",
            str(RECOVERY_ROOT),
            "--artifact-id",
            "campaign-state",
            "--campaign-id",
            CAMPAIGN_ID,
            "--project-commit",
            PROJECT_COMMIT,
            "--metadata-json",
            json.dumps({"label": label, "target": CAMPAIGN_TARGET}),
        ],
    )
    run_colab_command(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
            "status",
            "--output",
            str(STATUS_PATH),
            "--values-json",
            json.dumps(
                {
                    "state": "checkpointed",
                    "stage": label,
                    "target": CAMPAIGN_TARGET,
                    "project_commit": PROJECT_COMMIT,
                }
            ),
        ],
    )
    print("Drive state/checkpoint generation updated:", label)

In [ ]:
# One pinned hermetic runtime; no automatic compatibility fallback.
FAILURE_REPORTER.set_stage("runtime-hermetic-install")
if not RUNTIME_REQUIRED:
    RUNTIME_PYTHON = Path(sys.executable)
    MMSEG_ROOT = PROJECT_ROOT
    print("Bu hedef CUDA/MMCV/MMSeg gerektirmiyor; ağır runtime kurulumu atlandı.")
elif LOCAL_TEST_MODE:
    RUNTIME_PYTHON = Path(sys.executable)
    MMSEG_ROOT = PROJECT_ROOT
    print("LOCAL_TEST_MODE: CUDA compatibility installation skipped.")
else:
    import shutil

    import torch

    from edgeguard.rescue.colab_performance import environment_signature

    runtime_cache_identity = environment_signature(torch)
    runtime_cache_identity["framework_config_sha256"] = sha256_file(
        PROJECT_ROOT / "configs/training/segmentation/framework_mmseg.yaml"
    )
    runtime_cache_key = sha256_payload(runtime_cache_identity)
    runtime_cache_store = DRIVE_ROOT / "EdgeGuard/runtime_cache/v2"
    runtime_cache_artifact = f"uv-cache-{runtime_cache_key[:20]}"
    runtime_cache_archive = CONTENT_ROOT / "edgeguard-runtime-cache.tar"
    runtime_cache_root = CONTENT_ROOT / "edgeguard-cache"
    runtime_cache_pointer = runtime_cache_store / "pointers" / f"{runtime_cache_artifact}.json"
    cache_was_restored = False
    if runtime_cache_pointer.is_file() and not runtime_cache_root.exists():
        run_colab_command(
            [
                sys.executable,
                str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
                "restore",
                "--store-root",
                str(runtime_cache_store),
                "--artifact-id",
                runtime_cache_artifact,
                "--destination",
                str(runtime_cache_archive),
            ],
        )
        run_colab_command(
            [
                sys.executable,
                str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
                "restore-state",
                "--archive",
                str(runtime_cache_archive),
                "--destination",
                str(runtime_cache_root),
            ],
        )
        cache_was_restored = True
        print("Doğrulanmış runtime wheel/source cache /content alanına geri yüklendi.")

    compatibility_evidence = CONTENT_ROOT / "edgeguard-evidence"
    compatibility_logs = CONTENT_ROOT / "edgeguard-logs"
    drive_runtime_evidence = CAMPAIGN_ROOT / "runtime-compatibility"

    def persist_compatibility_evidence() -> None:
        drive_runtime_evidence.mkdir(parents=True, exist_ok=True)
        roots = ((compatibility_evidence, "evidence"), (compatibility_logs, "logs"))
        for root, label in roots:
            if not root.is_dir():
                continue
            for source in sorted(root.rglob("*")):
                if not source.is_file() or source.stat().st_size > 5 * 1024**2:
                    continue
                if source.suffix not in {".json", ".zip", ".log"}:
                    continue
                target = drive_runtime_evidence / label / source.relative_to(root)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, target)

    install = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/train/install_semantic_stack.py"),
        "--config",
        str(PROJECT_ROOT / "configs/training/segmentation/framework_mmseg.yaml"),
        "--project-root",
        str(PROJECT_ROOT),
        "--project-commit",
        PROJECT_COMMIT,
        "--config-root",
        str(PROJECT_ROOT / "configs/training/segmentation"),
        "--runtime-root",
        str(CONTENT_ROOT / "edgeguard-runtime"),
        "--checkout-root",
        str(CONTENT_ROOT / "edgeguard-checkouts"),
        "--evidence-root",
        str(CONTENT_ROOT / "edgeguard-evidence"),
        "--log-root",
        str(CONTENT_ROOT / "edgeguard-logs"),
        "--cache-root",
        str(CONTENT_ROOT / "edgeguard-cache"),
        "--data-root",
        str(CITYSCAPES_ROOT),
        "--execute",
    ]
    try:
        run_colab_command(install)
    except BaseException:
        persist_compatibility_evidence()
        raise
    if not cache_was_restored and (runtime_cache_root / "uv").is_dir():
        run_colab_command(
            [
                sys.executable,
                str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
                "pack-state",
                "--work-root",
                str(runtime_cache_root),
                "--output",
                str(runtime_cache_archive),
                "--include",
                "uv",
                "--uncompressed",
            ],
        )
        run_colab_command(
            [
                sys.executable,
                str(PROJECT_ROOT / "scripts/manage_colab_recovery.py"),
                "publish",
                "--source",
                str(runtime_cache_archive),
                "--store-root",
                str(runtime_cache_store),
                "--artifact-id",
                runtime_cache_artifact,
                "--campaign-id",
                CAMPAIGN_ID,
                "--project-commit",
                PROJECT_COMMIT,
                "--metadata-json",
                json.dumps(runtime_cache_identity),
            ],
        )
    runtime_report = CONTENT_ROOT / "edgeguard-evidence/resolved-runtime.json"
    run_colab_command(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts/resolve_colab_runtime.py"),
            "--receipt",
            str(CONTENT_ROOT / "edgeguard-evidence/runtime_receipt.json"),
            "--project-commit",
            PROJECT_COMMIT,
            "--output",
            str(runtime_report),
        ],
    )
    runtime = json.loads(runtime_report.read_text())
    RUNTIME_PYTHON = Path(runtime["interpreter"])
    MMSEG_ROOT = Path(runtime["mmseg_root"])
    persist_compatibility_evidence()
    work_runtime_evidence = WORK_ROOT / "reports/runtime-compatibility"
    work_runtime_evidence.mkdir(parents=True, exist_ok=True)
    for source in (compatibility_evidence / "runtime_receipt.json", runtime_report):
        shutil.copy2(source, work_runtime_evidence / source.name)
    training_profile = WORK_ROOT / "reports/runtime-compatibility/training-profile.json"
    run_colab_command(
        [
            str(RUNTIME_PYTHON),
            str(PROJECT_ROOT / "scripts/resolve_training_profile.py"),
            "--output",
            str(training_profile),
        ],
    )

In [ ]:
FAILURE_REPORTER.set_stage("dataset-audit-and-freeze")
AUDIT_ROOT = WORK_ROOT / "audit/cityscapes"
MANIFEST_ROOT = WORK_ROOT / "manifests"
DATA_MANIFESTS = [
    MANIFEST_ROOT / f"{dataset}.frozen.json" for dataset in SCIENTIFIC_SOURCE_DATASETS
]
STATS_ROOT = WORK_ROOT / "multidomain-statistics"
WEIGHTS = STATS_ROOT / "class_weights.json"
RARE = STATS_ROOT / "rare_classes.json"

if LOCAL_TEST_MODE:
    print("LOCAL_TEST_MODE: real dataset audit skipped.")
elif "audit" not in PLANNED_STAGES:
    print("Review hedefi: dataset staging ve audit atlandı.")
else:

    def preparation_identity(dataset_root):
        receipt = dataset_root / "preparation_receipt.json"
        return sha256_file(receipt) if receipt.is_file() else "legacy-without-receipt"

    cityscapes_audit = AUDIT_ROOT / "dataset_audit"
    city_inputs = {"preparation": preparation_identity(CITYSCAPES_ROOT)}
    if not completion_is_valid(cityscapes_audit, expected_inputs=city_inputs):
        quarantine_incomplete(cityscapes_audit, expected_inputs=city_inputs)
        run_colab_command(
            [
                str(RUNTIME_PYTHON),
                str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                "--dataset-root",
                str(CITYSCAPES_ROOT),
                "--output-root",
                str(AUDIT_ROOT),
            ]
        )
        write_completion_receipt(
            cityscapes_audit,
            artifact_type="cityscapes_audit",
            required_paths=["summary.json", "dataset_manifest.candidate.json", "CSF-SPLIT-D.json"],
            inputs=city_inputs,
            metadata={"dataset": "cityscapes"},
        )
        sync_work_snapshot("audit-cityscapes")
    audit = json.loads((cityscapes_audit / "summary.json").read_text())
    if not audit["audit_passed"]:
        raise RuntimeError("Cityscapes audit failed; scientific training is blocked")

    MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
    if RUN_MULTIDOMAIN_AUDIT:
        source_manifest_candidates = {
            "cityscapes": cityscapes_audit / "dataset_manifest.candidate.json"
        }
        provisional = PROVISIONAL_ENGINEERING_DATASETS if STAGE_PROVISIONAL_BDD else []
        for dataset in [*provisional, *SECONDARY_SCIENTIFIC_DATASETS]:
            dataset_root = DATASET_ROOTS[dataset]
            destination = WORK_ROOT / "audit" / dataset
            report_root = destination / f"{dataset}_audit"
            audit_inputs = {
                "preparation": preparation_identity(dataset_root),
                "ontology": sha256_file(PROJECT_ROOT / "configs/dataset/semantic_ontology_v2.yaml"),
            }
            audit_inputs.update(
                {
                    f"source_manifest:{source_dataset}": sha256_file(source_manifest)
                    for source_dataset, source_manifest in source_manifest_candidates.items()
                }
            )
            if completion_is_valid(report_root, expected_inputs=audit_inputs):
                if dataset in SECONDARY_SCIENTIFIC_DATASETS:
                    source_manifest_candidates[dataset] = (
                        report_root / "dataset_manifest.candidate.json"
                    )
                continue
            quarantine_incomplete(report_root, expected_inputs=audit_inputs)
            command = [
                str(RUNTIME_PYTHON),
                str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                "--dataset",
                dataset,
                "--dataset-root",
                str(dataset_root),
                "--output-root",
                str(destination),
            ]
            for source_manifest in source_manifest_candidates.values():
                command.extend(["--source-manifest", str(source_manifest)])
            if dataset == "idd20k":
                command.extend(
                    [
                        "--checkpoint-root",
                        str(CAMPAIGN_ROOT / "state/audit-catalog"),
                        "--quarantine-invalid-source-samples",
                    ]
                )
            audit_process = run_colab_command(command, check=False)
            if audit_process.returncode not in {0, 2}:
                raise RuntimeError(
                    f"{dataset} audit infrastructure failed with exit code "
                    f"{audit_process.returncode}"
                )
            if audit_process.returncode == 2:
                review_root = CAMPAIGN_ROOT / "reports/data-review" / dataset
                review_root.mkdir(parents=True, exist_ok=True)
                for name in (
                    "summary.json",
                    "invalid_samples.json",
                    "dataset_manifest.candidate.json",
                ):
                    source = report_root / name
                    if source.is_file():
                        shutil.copy2(source, review_root / name)
                review_status = {
                    "status": "data_review_required",
                    "dataset": dataset,
                    "report_root": str(review_root),
                    "message": (
                        "Audit found a fail-closed data contract violation. Training was not "
                        "started; review artifacts are persistent on Drive."
                    ),
                }
                (review_root / "status.json").write_text(
                    canonical_json(review_status) + "\n", encoding="utf-8"
                )
                sync_work_snapshot(f"audit-review-{dataset}")
                raise RuntimeError(canonical_json(review_status))
            write_completion_receipt(
                report_root,
                artifact_type="source_domain_audit",
                required_paths=[
                    "summary.json",
                    "invalid_samples.json",
                    "dataset_manifest.candidate.json",
                ],
                inputs=audit_inputs,
                metadata={"dataset": dataset},
            )
            if dataset in SECONDARY_SCIENTIFIC_DATASETS:
                source_manifest_candidates[dataset] = (
                    report_root / "dataset_manifest.candidate.json"
                )
            sync_work_snapshot(f"audit-{dataset}")

    if RUN_FREEZE:
        candidates = {"cityscapes": cityscapes_audit / "dataset_manifest.candidate.json"}
        for dataset in SECONDARY_SCIENTIFIC_DATASETS:
            candidates[dataset] = (
                WORK_ROOT / "audit" / dataset / f"{dataset}_audit/dataset_manifest.candidate.json"
            )
        for dataset, candidate in candidates.items():
            if not candidate.is_file():
                raise RuntimeError(f"Missing reviewed audit candidate for {dataset}")
            review_receipt = MANIFEST_REVIEW_RECEIPT_ROOT / f"{dataset}.review.json"
            if not review_receipt.is_file():
                raise PermissionError(
                    "Manifest freeze is closed until a human review receipt exists: "
                    f"{review_receipt}"
                )
            frozen = MANIFEST_ROOT / f"{dataset}.frozen.json"
            reuse = False
            if frozen.is_file():
                frozen_payload = json.loads(frozen.read_text())
                reuse = (
                    frozen_payload.get("approved_candidate_sha256") == sha256_file(candidate)
                    and frozen_payload.get("human_review_receipt_sha256")
                    == sha256_file(review_receipt)
                    and frozen_payload.get("project_commit") == PROJECT_COMMIT
                    and frozen_payload.get("campaign_id") == CAMPAIGN_ID
                )
            if not reuse:
                if frozen.exists():
                    quarantine = frozen.with_name(f"{frozen.name}.stale-{PROJECT_COMMIT[:12]}")
                    frozen.replace(quarantine)
                run_colab_command(
                    [
                        str(RUNTIME_PYTHON),
                        str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                        "--dataset",
                        dataset,
                        "--output-root",
                        str(MANIFEST_ROOT),
                        "--split-manifest",
                        str(candidate),
                        "--freeze-approved",
                        "--review-receipt",
                        str(review_receipt),
                        "--campaign-id",
                        CAMPAIGN_ID,
                        "--project-commit",
                        PROJECT_COMMIT,
                    ]
                )
        sync_work_snapshot("frozen-source-manifests")

    if RUN_SOURCE_VALIDATION_AUDIT:
        if not all(path.is_file() for path in DATA_MANIFESTS):
            raise RuntimeError("Official source validation requires all frozen source manifests")
        completed_final_runs = [
            path for path in (WORK_ROOT / "runs/final").glob("*/ce") if completion_is_valid(path)
        ]
        if len(completed_final_runs) < 3:
            raise RuntimeError(
                "Official validation remains sealed until CAMPAIGN_TARGET='final' completes"
            )
        manifest_set = sha256_payload(sorted(sha256_file(path) for path in DATA_MANIFESTS))
        for dataset in OFFICIAL_VALIDATION_DATASETS:
            destination = WORK_ROOT / "audit" / f"{dataset}-val"
            report_root = (
                destination / "dataset_audit"
                if dataset == "cityscapes"
                else destination / f"{dataset}_val_audit"
            )
            validation_inputs = {
                "training_manifests": manifest_set,
                "preparation": preparation_identity(DATASET_ROOTS[dataset]),
            }
            if completion_is_valid(report_root, expected_inputs=validation_inputs):
                continue
            quarantine_incomplete(report_root)
            command = [
                str(RUNTIME_PYTHON),
                str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                "--dataset",
                dataset,
                "--source-split",
                "val",
                "--dataset-root",
                str(DATASET_ROOTS[dataset]),
                "--output-root",
                str(destination),
            ]
            if dataset != "cityscapes":
                command.extend(["--checkpoint-root", str(CAMPAIGN_ROOT / "state/audit-catalog")])
            for manifest in DATA_MANIFESTS:
                command.extend(["--source-manifest", str(manifest)])
            run_colab_command(command)
            write_completion_receipt(
                report_root,
                artifact_type="official_source_validation_audit",
                required_paths=["summary.json", "dataset_manifest.candidate.json"],
                inputs=validation_inputs,
                metadata={"dataset": dataset},
            )
            sync_work_snapshot(f"official-validation-audit-{dataset}")

    if RUN_FREEZE_SOURCE_VALIDATION:
        validation_root = MANIFEST_ROOT / "official-validation"
        for dataset in OFFICIAL_VALIDATION_DATASETS:
            report_name = "dataset_audit" if dataset == "cityscapes" else f"{dataset}_val_audit"
            candidate = (
                WORK_ROOT
                / "audit"
                / f"{dataset}-val"
                / report_name
                / "dataset_manifest.candidate.json"
            )
            frozen = validation_root / f"{dataset}.frozen.json"
            review_receipt = MANIFEST_REVIEW_RECEIPT_ROOT / f"{dataset}-official-val.review.json"
            if not candidate.is_file() or not review_receipt.is_file():
                raise PermissionError(
                    "Official validation freeze requires candidate and review receipt for "
                    f"{dataset}"
                )
            reuse = False
            if frozen.is_file():
                frozen_payload = json.loads(frozen.read_text())
                reuse = (
                    frozen_payload.get("approved_candidate_sha256") == sha256_file(candidate)
                    and frozen_payload.get("human_review_receipt_sha256")
                    == sha256_file(review_receipt)
                    and frozen_payload.get("project_commit") == PROJECT_COMMIT
                )
            if not reuse:
                if frozen.exists():
                    quarantine = frozen.with_name(f"{frozen.name}.stale-{PROJECT_COMMIT[:12]}")
                    frozen.replace(quarantine)
                run_colab_command(
                    [
                        str(RUNTIME_PYTHON),
                        str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                        "--dataset",
                        dataset,
                        "--output-root",
                        str(validation_root),
                        "--split-manifest",
                        str(candidate),
                        "--freeze-approved",
                        "--review-receipt",
                        str(review_receipt),
                        "--campaign-id",
                        CAMPAIGN_ID,
                        "--project-commit",
                        PROJECT_COMMIT,
                    ]
                )
        sync_work_snapshot("frozen-official-validation-manifests")

    if all(path.is_file() for path in DATA_MANIFESTS):
        stats_inputs = {
            "dataset_manifest_set": sha256_payload(
                sorted(sha256_file(path) for path in DATA_MANIFESTS)
            )
        }
        if not completion_is_valid(STATS_ROOT, expected_inputs=stats_inputs):
            quarantine_incomplete(STATS_ROOT)
            command = [
                str(RUNTIME_PYTHON),
                str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                "--output-root",
                str(STATS_ROOT),
            ]
            for manifest in DATA_MANIFESTS:
                command.extend(["--data-manifest", str(manifest)])
            run_colab_command(command)
            write_completion_receipt(
                STATS_ROOT,
                artifact_type="multidomain_statistics",
                required_paths=["class_weights.json", "rare_classes.json", "summary.json"],
                inputs=stats_inputs,
            )
            sync_work_snapshot("multidomain-statistics")
    print(json.dumps(audit, indent=2))

In [ ]:
FAILURE_REPORTER.set_stage("resumable-python-orchestrator")
RUN_ROOT = WORK_ROOT / "runs"


def ensure_checkpoint(stage, model):
    return latest_checkpoint(RUN_ROOT / stage / model / "ce")


PIPELINE_TARGETS = {"smoke", "pilot", "screening", "hpo", "final", "evaluate", "export", "report"}
if LOCAL_TEST_MODE:
    print("LOCAL_TEST_MODE: hermetic GPU pipeline execution skipped.")
elif CAMPAIGN_TARGET in PIPELINE_TARGETS:
    if RUN_ACCEPT_RELEASE:
        if ACCEPTED_RELEASE.is_file():
            print("Existing accepted release will be hash-verified by the orchestrator.")
        elif not RELEASE_CANDIDATE.is_file() or not RELEASE_REVIEW_RECEIPT.is_file():
            raise PermissionError("Release acceptance requires candidate and human review receipt")
        else:
            run_colab_command(
                [
                    str(RUNTIME_PYTHON),
                    str(PROJECT_ROOT / "scripts/accept_colab_release.py"),
                    "--candidate",
                    str(RELEASE_CANDIDATE),
                    "--review-receipt",
                    str(RELEASE_REVIEW_RECEIPT),
                    "--output",
                    str(ACCEPTED_RELEASE),
                ]
            )
    if not all(path.is_file() for path in DATA_MANIFESTS):
        raise RuntimeError("All reviewed/frozen scientific source manifests are required")
    pipeline_command = [
        str(RUNTIME_PYTHON),
        str(PROJECT_ROOT / "scripts/colab_pipeline.py"),
        "run",
        "--target",
        CAMPAIGN_TARGET,
        "--project-root",
        str(PROJECT_ROOT),
        "--project-commit",
        PROJECT_COMMIT,
        "--runtime-receipt",
        str(CONTENT_ROOT / "edgeguard-evidence/runtime_receipt.json"),
        "--mmseg-root",
        str(MMSEG_ROOT),
        "--work-root",
        str(WORK_ROOT),
        "--recovery-root",
        str(RECOVERY_ROOT),
        "--config",
        str(PROJECT_ROOT / "configs/rescue/semantic_first.yaml"),
        "--rare-classes-file",
        str(RARE),
        "--class-weights-file",
        str(WEIGHTS),
    ]
    for manifest in DATA_MANIFESTS:
        pipeline_command.extend(["--data-manifest", str(manifest)])
    candidate_table = WORK_ROOT / "reports/screening/candidate_table.json"
    if CAMPAIGN_TARGET in {"hpo", "final", "evaluate", "export", "report"}:
        if not candidate_table.is_file():
            raise RuntimeError("HPO/final target requires the reviewed screening candidate table")
        pipeline_command.extend(["--candidate-table", str(candidate_table)])
    if CAMPAIGN_TARGET in {"final", "evaluate", "export", "report"}:
        if len(FINAL_MODELS) != 3:
            raise RuntimeError("Final and accepted-release targets require three frozen finalists")
        for model in FINAL_MODELS:
            pipeline_command.extend(["--final-model", model])
        pipeline_command.extend(["--ablation-model", FINAL_MODELS[0]])
    if CAMPAIGN_TARGET in {"evaluate", "export", "report"}:
        if not ACCEPTED_RELEASE.is_file():
            raise PermissionError("Accepted-release targets require ACCEPTED_RELEASE")
        pipeline_command.extend(["--accepted-release", str(ACCEPTED_RELEASE)])
        validation_root = WORK_ROOT / "manifests/official-validation"
        for dataset in OFFICIAL_VALIDATION_DATASETS:
            manifest = validation_root / f"{dataset}.frozen.json"
            if not manifest.is_file():
                raise PermissionError(
                    f"Accepted-release targets require frozen official validation: {manifest}"
                )
            pipeline_command.extend(["--evaluation-manifest", str(manifest)])
    run_colab_command(pipeline_command)
    sync_work_snapshot(f"pipeline-{CAMPAIGN_TARGET}")
else:
    print("Bu hedef GPU eğitim orchestrator'ını gerektirmiyor.")

In [ ]:
FAILURE_REPORTER.set_stage("pipeline-artifact-review")
pipeline_state = WORK_ROOT / "pipeline-v2"
if pipeline_state.is_dir():
    completed = sorted(path.parent.name for path in pipeline_state.glob("*/completion.json"))
    print("Hash-doğrulanmış pipeline fazları:", completed)
else:
    print("Henüz bir v2 pipeline fazı tamamlanmadı.")

In [ ]:
FAILURE_REPORTER.set_stage("final-training-owned-by-orchestrator")
if CAMPAIGN_TARGET == "final":
    print(
        "Final eğitim, HPO parametreleri ve CE/weighted-CE ablation "
        "orchestrator tarafından işlendi."
    )

In [ ]:
FAILURE_REPORTER.set_stage("final-calibration-and-evaluation")
exec(FINAL_PROTOCOL_CODE)

In [ ]:
# Temperature sonrası source-only shift referansı; external veri eşik üretimine giremez.
FAILURE_REPORTER.set_stage("shift-calibration-and-perception-preview")
if RUN_SHIFT_CALIBRATION:
    for model in FINAL_MODELS:
        run_dir = RUN_ROOT / "final" / model / "ce"
        checkpoint = latest_checkpoint(run_dir)
        temperature = WORK_ROOT / "calibration" / model / "global-temperature.json"
        if not temperature.is_file():
            raise RuntimeError(f"Missing global temperature for {model}")
        source_summaries = []
        for dataset, manifest in zip(SCIENTIFIC_SOURCE_DATASETS, DATA_MANIFESTS, strict=True):
            target = WORK_ROOT / "evaluation/shift-calibration" / model / dataset
            if not target.exists():
                run_colab_command(
                    [
                        str(RUNTIME_PYTHON),
                        str(PROJECT_ROOT / "scripts/evaluate.py"),
                        "run",
                        "--resolved-config",
                        str(run_dir / "resolved.py"),
                        "--checkpoint",
                        str(checkpoint),
                        "--dataset",
                        dataset,
                        "--dataset-manifest",
                        str(manifest),
                        "--role",
                        "train_calibration",
                        "--temperature-file",
                        str(temperature),
                        "--output-dir",
                        str(target),
                    ]
                )
            source_summaries.append(target / "frame_uncertainty.json")
        shift_reference = WORK_ROOT / "calibration" / model / "source-shift-reference.json"
        if not shift_reference.exists():
            command = [
                str(RUNTIME_PYTHON),
                str(PROJECT_ROOT / "scripts/evaluate.py"),
                "calibrate-shift",
                "--checkpoint",
                str(checkpoint),
                "--output",
                str(shift_reference),
            ]
            for summary in source_summaries:
                command.extend(["--summary", str(summary)])
            for manifest in DATA_MANIFESTS:
                command.extend(["--data-manifest", str(manifest)])
            run_colab_command(command)
        if RUN_ACDC:
            source_final = [
                WORK_ROOT / "evaluation/final" / model / dataset / "frame_uncertainty.json"
                for dataset in SCIENTIFIC_SOURCE_DATASETS
            ]
            for condition in ("fog", "night", "rain", "snow"):
                external_summary = (
                    WORK_ROOT / "evaluation/acdc" / model / condition / "frame_uncertainty.json"
                )
                output = WORK_ROOT / "evaluation/shift" / model / f"acdc-{condition}.json"
                if (
                    external_summary.is_file()
                    and all(path.is_file() for path in source_final)
                    and not output.exists()
                ):
                    command = [
                        str(RUNTIME_PYTHON),
                        str(PROJECT_ROOT / "scripts/evaluate.py"),
                        "evaluate-shift",
                        "--reference",
                        str(shift_reference),
                        "--external-summary",
                        str(external_summary),
                        "--output",
                        str(output),
                    ]
                    for summary in source_final:
                        command.extend(["--source-summary", str(summary)])
                    run_colab_command(command)
        if RUN_PERCEPTION_PREVIEW:
            onnx_model = WORK_ROOT / "exports/final" / model / f"{model}.onnx"
            if not PREVIEW_IMAGE.is_file() or not onnx_model.is_file():
                raise RuntimeError("Perception preview requires PREVIEW_IMAGE and final ONNX")
            preview = WORK_ROOT / "preview" / model
            if not preview.exists():
                run_colab_command(
                    [
                        str(RUNTIME_PYTHON),
                        str(PROJECT_ROOT / "scripts/predict.py"),
                        "--image",
                        str(PREVIEW_IMAGE),
                        "--model",
                        str(onnx_model),
                        "--output-dir",
                        str(preview),
                        "--emit-regions",
                        "--emit-risk",
                        "--shift-reference",
                        str(shift_reference),
                    ]
                )

In [ ]:
FAILURE_REPORTER.set_stage("output-snapshot-and-review-package")
sync_work_snapshot("cell-complete")
if CREATE_REVIEW_PACKAGE and WORK_ROOT.is_dir():
    review_root = DRIVE_ROOT / "EdgeGuard/review_packages"
    review_root.mkdir(parents=True, exist_ok=True)
    review_zip = review_root / f"{CAMPAIGN_ID}-{PROJECT_COMMIT[:12]}-{RUN_STAGE}-review.zip"
    run_colab_command(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts/package_colab_outputs.py"),
            "review",
            "--source-root",
            str(WORK_ROOT),
            "--output",
            str(review_zip),
            "--campaign-id",
            CAMPAIGN_ID,
            "--project-commit",
            PROJECT_COMMIT,
        ],
    )
    print("İnceleme paketi:", review_zip)
    if DOWNLOAD_REVIEW_PACKAGE and not LOCAL_TEST_MODE:
        from google.colab import files

        files.download(str(review_zip))
print("Demo command: streamlit run", PROJECT_ROOT / "app.py")
print("Jetson handoff: scripts/jetson/build_tensorrt.py then scripts/jetson/benchmark.py")
print("Hata raporu kökü:", FAILURE_REPORTER.output_root)
if DOWNLOAD_LATEST_FAILURE_REPORT:
    latest_failure = FAILURE_REPORTER.latest_package()
    if latest_failure is None:
        raise RuntimeError("İndirilecek hata paketi bulunamadı")
    if not LOCAL_TEST_MODE:
        from google.colab import files

        files.download(str(latest_failure))
    print("Hata paketi:", latest_failure)